# XWC S6E10 V11 Masked Signal — submitted predictions

This notebook exports the author's supplied prediction file for official evaluation in [Playground Series S6E10](https://www.kaggle.com/competitions/playground-series-s6e10). It checks the file hash, test IDs, row count and prediction range before producing `submission.csv`.

The original V11 training code is included in the attached [artifacts dataset](https://www.kaggle.com/datasets/dennis6886/xwc-s6e10-v11-submitted-artifacts). This version runs the supplied predictions; it does not rerun the stopped local training. No OOF or CV score is claimed for this supplied CSV.

## Validate the supplied artifact

The input SHA-256 is fixed before execution. The output preserves the exact original CSV bytes.

In [1]:
from pathlib import Path
import hashlib
import json
import shutil
import numpy as np
import pandas as pd

EXPECTED_SHA256 = "c5d1158d15172ffd632b20ebbd651523edd5605cb2b91c11a3dcb43f1704c5d2"
EXPECTED_ROWS = 299844
input_root = Path("/kaggle/input")
candidates = sorted(input_root.rglob("submission.csv"))
matches = [p for p in candidates if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_SHA256]
assert len(matches) == 1, "Expected exactly one input matching the supplied CSV hash"
source = matches[0]
pred = pd.read_csv(source)
assert list(pred) == ["id", "satisfaction"]
assert len(pred) == EXPECTED_ROWS and pred["id"].is_unique
assert pred.notna().all().all()
assert np.isfinite(pred["satisfaction"]).all()
assert pred["satisfaction"].between(0, 1).all()
samples = list(input_root.rglob("sample_submission.csv"))
assert samples, "Competition sample submission must be attached"
assert any(pred["id"].equals(pd.read_csv(p)["id"]) for p in samples), "ID order mismatch"
validation = {"rows": len(pred), "columns": list(pred), "sha256": EXPECTED_SHA256,
              "finite": True, "range": [float(pred.satisfaction.min()), float(pred.satisfaction.max())],
              "exact_sample_id_order": True, "cv": None,
              "operation": "Byte-for-byte export of the user-supplied prediction file"}
print(json.dumps(validation, indent=2))
pred.head()


{
  "rows": 299844,
  "columns": [
    "id",
    "satisfaction"
  ],
  "sha256": "c5d1158d15172ffd632b20ebbd651523edd5605cb2b91c11a3dcb43f1704c5d2",
  "finite": true,
  "range": [
    3.3350675684689366e-06,
    1.0
  ],
  "exact_sample_id_order": true,
  "cv": null,
  "operation": "Byte-for-byte export of the user-supplied prediction file"
}


,id,satisfaction
0,699635,0.522205
1,699636,0.582921
2,699637,0.648521
3,699638,0.468981
4,699639,0.296501


In [2]:
output = Path("/kaggle/working/submission.csv")
shutil.copyfile(source, output)
assert hashlib.sha256(output.read_bytes()).hexdigest() == EXPECTED_SHA256
Path("/kaggle/working/artifact_validation.json").write_text(json.dumps(validation, indent=2))
print("Created", output.name, "with", len(pred), "rows; original bytes preserved.")


Created submission.csv with 299844 rows; original bytes preserved.


## Attribution

The original archive credits Areeb Ali, tkms0816, MAYANK RAJ, Subhanjan Das and the public OOF/model authors credited by those upstream notebooks. See `NOTICE.md` in the attached dataset for attribution and applicable restrictions.

The official competition score is determined by Kaggle after submission of this notebook version's output.